# A Predictive Model for US Airline Flight Delays

### <span style="color:chocolate"> Project Description </span>

Flight delay prediction is an important research field, as flight delays are the primary concern of aviation
stakeholders. Our project aims to identify the predictors of flight delays before they happen, improve
travelers experience, and help airlines shift from reactive damage control to proactive delay management.

---
### <span style="color:chocolate">Import libraries</span>

In [23]:
import numpy as np
import pandas as pd
import glob
import os

---
### <span style="color:chocolate">Data Ingestion</span>

Reading the data from 2022 to 2026 to the data frame downloaded from the following data sources:
- Flight data: Bureau of transportation stats https://www.transtats.bts.gov/ontime/
- Weather data (New with 2026 data): https://www.ncei.noaa.gov/access/search/data-search/global-historical-climatology-network-hourly
- Aircraft data: https://www.faa.gov/licenses_certificates/aircraft_certification/aircraft_registry/releasable_aircraft_download?utm_source=chatgpt.com

In [24]:
# Read in all weather data in the psv files to the data frame and indicate 'I' as the delimiter.
weather_files_path = "Data/*.psv"
all_weather_files = glob.glob(weather_files_path)
weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)

# Read in all flight data to the data frame. The first 7 lines that contains the data information are skipped.
flights_files_path = "Data/DFW_Arrivals_*.csv"
all_flights_files = glob.glob(flights_files_path)
flights = pd.concat((pd.read_csv(f, skiprows=7) for f in all_flights_files), ignore_index=True)

# Read in all aircraft data to the data frame.
aircraft = pd.read_csv('Data/ReleasableAircraft/MASTER.txt', sep=',')

/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_67700/3261188870.py:4: DtypeWarning: Columns (43,55,77,80,82,97,103,128,130,133,217,223,229,241,247,253) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)
/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_67700/3261188870.py:4: DtypeWarning: Columns (13,19,43,55,77,80,82,91,97,103,133,217,223,229,241,247,253) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f in all_weather_files), ignore_index=True)
/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_67700/3261188870.py:4: DtypeWarning: Columns (61,67,71,74,76,77,80,82,89,92,94,95,98,100,101,104,106,113,116,118,119,122,124,240,246,252,320,322) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.concat((pd.read_csv(f, sep="|") for f i

---
#### <span style="color:chocolate">Data Preprocessing</span>



In [25]:
# Print out the shape and data type of the weather data frame.
print("The shape of the weather data frame:", weather.shape)
print("The data type of the weather data frame", weather.dtypes)
weather.head()

The shape of the weather data frame: (84896, 329)
The data type of the weather data frame STATION                   object
Station_name              object
DATE                      object
Year                       int64
Month                      int64
                          ...   
REM_Measurement_Code     float64
REM_Quality_Code         float64
REM_Report_Type           object
REM_Source_Code          float64
REM_Source_Station_ID     object
Length: 329, dtype: object


,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00003927,DAL-FTW WSCMO AP,2022-01-01T00:00:00,2022,1,1,0,0,32.8975,-97.0219,...,NaN,NaN,NaN,NaN,SYN08672259 12966 81607 10211 20167 39831 4003...,NaN,NaN,FM12,223.0,ICAO-KDFW
1,USW00003927,DAL-FTW WSCMO AP,2022-01-01T00:53:00,2022,1,1,0,53,32.8975,-97.0219,...,NaN,NaN,NaN,NaN,MET12612/31/21 18:53:03 METAR KDFW 010053Z 160...,NaN,NaN,FM15,343.0,722590-03927
2,USW00003927,DAL-FTW WSCMO AP,2022-01-01T01:53:00,2022,1,1,1,53,32.8975,-97.0219,...,NaN,NaN,NaN,NaN,MET11912/31/21 19:53:03 METAR KDFW 010153Z 110...,NaN,NaN,FM15,343.0,722590-03927
3,USW00003927,DAL-FTW WSCMO AP,2022-01-01T02:53:00,2022,1,1,2,53,32.8975,-97.0219,...,NaN,NaN,NaN,NaN,MET11612/31/21 20:53:03 METAR KDFW 010253Z 120...,NaN,NaN,FM15,343.0,722590-03927
4,USW00003927,DAL-FTW WSCMO AP,2022-01-01T03:00:00,2022,1,1,3,0,32.8975,-97.0219,...,NaN,NaN,NaN,NaN,SYN07072259 12562 81203 10194 20172 39834 4003...,NaN,NaN,FM12,223.0,ICAO-KDFW


In [26]:
# Print out the shape and data type of the flights data frame.
print("The shape of the flights data frame:", flights.shape)
print("The data type of the flights data frame:", flights.dtypes)
flights.head()

The shape of the flights data frame: (758931, 17)
The data type of the flights data frame: Carrier Code                                 object
Date (MM/DD/YYYY)                            object
Flight Number                               float64
Tail Number                                  object
Origin Airport                               object
Scheduled Arrival Time                       object
Actual Arrival Time                          object
Scheduled Elapsed Time (Minutes)            float64
Actual Elapsed Time (Minutes)               float64
Arrival Delay (Minutes)                     float64
Wheels-on Time                               object
Taxi-In time (Minutes)                      float64
Delay Carrier (Minutes)                     float64
Delay Weather (Minutes)                     float64
Delay National Aviation System (Minutes)    float64
Delay Security (Minutes)                    float64
Delay Late Aircraft Arrival (Minutes)       float64
dtype: object


,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Scheduled Arrival Time,Actual Arrival Time,Scheduled Elapsed Time (Minutes),Actual Elapsed Time (Minutes),Arrival Delay (Minutes),Wheels-on Time,Taxi-In time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,DL,01/01/2022,531.0,N310DU,LGA,17:19,17:20,259.0,264.0,1.0,17:03,17.0,0.0,0.0,0.0,0.0,0.0
1,DL,01/01/2022,561.0,N307DU,LGA,11:57,11:35,262.0,246.0,-22.0,11:26,9.0,0.0,0.0,0.0,0.0,0.0
2,DL,01/01/2022,1036.0,N106DU,JFK,19:33,19:55,253.0,266.0,22.0,19:38,17.0,9.0,0.0,13.0,0.0,0.0
3,DL,01/01/2022,1041.0,N344NW,LAX,13:50,13:26,180.0,160.0,-24.0,13:19,7.0,0.0,0.0,0.0,0.0,0.0
4,DL,01/01/2022,1094.0,N326US,LAX,21:58,00:00,178.0,0.0,0.0,00:00,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# Print out the shape and data type of the aircraft data frame.
print("The shape of the aircraft data frame:", n_num.shape)
print("The data type of the aircraft data frame:", aircraft.dtypes)
aircraft.head()

The shape of the aircraft data frame: (313680, 35)
The data type of the aircraft data frame: N-NUMBER             object
SERIAL NUMBER        object
MFR MDL CODE         object
ENG MFR MDL          object
YEAR MFR             object
TYPE REGISTRANT      object
NAME                 object
STREET               object
STREET2              object
CITY                 object
STATE                object
ZIP CODE             object
REGION               object
COUNTY               object
COUNTRY              object
LAST ACTION DATE      int64
CERT ISSUE DATE      object
CERTIFICATION        object
TYPE AIRCRAFT        object
TYPE ENGINE           int64
STATUS CODE          object
MODE S CODE           int64
FRACT OWNER          object
AIR WORTH DATE       object
OTHER NAMES(1)       object
OTHER NAMES(2)       object
OTHER NAMES(3)       object
OTHER NAMES(4)       object
OTHER NAMES(5)       object
EXPIRATION DATE      object
UNIQUE ID             int64
KIT MFR              object
 KIT MODEL 

,N-NUMBER,SERIAL NUMBER,MFR MDL CODE,ENG MFR MDL,YEAR MFR,TYPE REGISTRANT,NAME,STREET,STREET2,CITY,...,OTHER NAMES(2),OTHER NAMES(3),OTHER NAMES(4),OTHER NAMES(5),EXPIRATION DATE,UNIQUE ID,KIT MFR,KIT MODEL,MODE S CODE HEX,Unnamed: 34
0,100,5334,7100510,17003,1940,1,BENE MARY D ...,PO BOX 329,,KETCHUM,...,...,...,...,...,20270430,600060,,,A004B3,NaN
1,10000,10000,2130004,,,7,9AT LLC ...,511 WEDGEWOOD AVE,,NASHVILLE,...,...,...,...,...,20310831,1443200,,,A00725,NaN
2,10001,A28,9601202,67007,1928,1,STOOS ROBERT A ...,PO BOX 1056,,LAKELAND,...,...,...,...,...,20290228,432072,,,A00726,NaN
3,10004,T18208245,2072738,,,7,ETOS AIR LLC ...,PO BOX 288,,NEW LONDON,...,...,...,...,...,20290331,102879,,,A00729,NaN
4,10006,BG-72,1152020,17026,1955,1,COUTCHES ROBERT HERCULES DBA ...,550 AIRWAY BLVD,,LIVERMORE,...,...,...,...,...,20280229,480110,,,A0072B,NaN


In [28]:
# Convert the "DATE" column in weather data frame to datetime object.
weather['Weather_Datetime'] = pd.to_datetime(weather['DATE'])
# Combine the "Date (MM/DD/YYYY)" and "Scheduled Arrival Time" columns into a single column for joining the weather data frame.
flights['Full_Time_Str'] = flights['Date (MM/DD/YYYY)'] + ' ' + flights['Scheduled Arrival Time']
# Convert the "Full_Time_Str" column created above to datatime object and round it to the nearest hour to match the time in weather data frame.
flights['Flight_Hourly'] = pd.to_datetime(flights['Full_Time_Str']).dt.round('h')
# Remove the alphabet 'N' from the "Tail Number" column in flights data frame to match the "N_Num" in aircraft data frame.
flights['Join_N_Num'] = flights['Tail Number'].str.lstrip('N')

# Join the flights and weather data frames together by using left join and the data time as key.
merged_df = pd.merge(
    flights,
    weather,
    left_on='Flight_Hourly',
    right_on='Weather_Datetime',
    how='left'
)

# Join the flights and aircraft data frames together by using left join and the data time as key.
final_df = pd.merge(
    merged_df,
    aircraft,
    left_on='Join_N_Num',
    right_on='N-NUMBER',
    how='left'
)

# Delete the columns that created above to join the data frames.
final_df = final_df.drop(columns=['Full_Time_Str', 'Flight_Hourly', 'Weather_Datetime', 'Join_N_Num'])

/var/folders/js/vbq1y5_96hg79pndsb_1rg1m0000gn/T/ipykernel_67700/2997416326.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weather['Weather_Datetime'] = pd.to_datetime(weather['DATE'])


,carrier_code,date_(mm/dd/yyyy),flight_number,tail_number,origin_airport,scheduled_arrival_time,actual_arrival_time,scheduled_elapsed_time_(minutes),actual_elapsed_time_(minutes),arrival_delay_(minutes),...,other_names(2),other_names(3),other_names(4),other_names(5),expiration_date,unique_id,kit_mfr,kit_model,mode_s_code_hex,unnamed:_34
0,DL,01/01/2022,531.0,N310DU,LGA,17:19,17:20,259.0,264.0,1.0,...,...,...,...,...,20281231,1355999.0,,,A3472F,NaN
1,DL,01/01/2022,561.0,N307DU,LGA,11:57,11:35,262.0,246.0,-22.0,...,...,...,...,...,20280131,1334505.0,,,A339B1,NaN
2,DL,01/01/2022,1036.0,N106DU,JFK,19:33,19:55,253.0,266.0,22.0,...,...,...,...,...,20270930,1291625.0,,,A01B5C,NaN
3,DL,01/01/2022,1041.0,N344NW,LAX,13:50,13:26,180.0,160.0,-24.0,...,...,...,...,...,20290630,924212.0,,,A3CD6B,NaN
4,DL,01/01/2022,1094.0,N326US,LAX,21:58,00:00,178.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Print out the shape and data type of the weather data frame.
print("The shape of the final data frame:", weather.shape)
print("The data type of the final data frame", weather.dtypes)
final_df.head()

The shape of the final data frame: (84896, 330)
The data type of the final data frame STATION                          object
Station_name                     object
DATE                             object
Year                              int64
Month                             int64
                              ...      
REM_Quality_Code                float64
REM_Report_Type                  object
REM_Source_Code                 float64
REM_Source_Station_ID            object
Weather_Datetime         datetime64[ns]
Length: 330, dtype: object


,carrier_code,date_(mm/dd/yyyy),flight_number,tail_number,origin_airport,scheduled_arrival_time,actual_arrival_time,scheduled_elapsed_time_(minutes),actual_elapsed_time_(minutes),arrival_delay_(minutes),...,other_names(2),other_names(3),other_names(4),other_names(5),expiration_date,unique_id,kit_mfr,kit_model,mode_s_code_hex,unnamed:_34
0,DL,01/01/2022,531.0,N310DU,LGA,17:19,17:20,259.0,264.0,1.0,...,...,...,...,...,20281231,1355999.0,,,A3472F,NaN
1,DL,01/01/2022,561.0,N307DU,LGA,11:57,11:35,262.0,246.0,-22.0,...,...,...,...,...,20280131,1334505.0,,,A339B1,NaN
2,DL,01/01/2022,1036.0,N106DU,JFK,19:33,19:55,253.0,266.0,22.0,...,...,...,...,...,20270930,1291625.0,,,A01B5C,NaN
3,DL,01/01/2022,1041.0,N344NW,LAX,13:50,13:26,180.0,160.0,-24.0,...,...,...,...,...,20290630,924212.0,,,A3CD6B,NaN
4,DL,01/01/2022,1094.0,N326US,LAX,21:58,00:00,178.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Replace the space in the column names with underline.
final_df.columns = final_df.columns.str.strip().str.lower().str.replace(' ', '_')

---
#### <span style="color:chocolate">Exploratory Data Analysis (EDA)</span>

Create a scatterplot matrix to visualize the pair-wise correlations between different features and outcome in the (X_train_std, Y_train_std) data.

Create a correlation matrix in the form of a heatmap to visualize the linear relationships between different features and outcome in the (X_train_std, Y_train_std) data

---
#### <span style="color:chocolate">Modeling</span>

Baseline Model

Improvement over Baseline

---
#### <span style="color:chocolate">Hyperparameter tuning</span>

Fine-tune the learning rate and number of epochs hyperparameters of model_tf to determine the setup that yields the most optimal generalization performance.

---
#### <span style="color:chocolate">Evaluation and Generalization:</span>

Evaluate the optimized (tuned) model on the test data.

---
#### <span style="color:chocolate">Analysis</span>

TBA

---
#### <span style="color:chocolate">TBA</span>

---
#### <span style="color:chocolate">TBA</span>

TBA

---
#### <span style="color:chocolate">TBA</span>

TBA

----
#### <span style="color:chocolate">TBA</span>

TBA